In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 1. 데이터 가져오기

In [11]:
import pandas as pd
df = pd.read_csv('C:/ai_x/source/01_python/data/전국평당분양가격(결측보완).csv', encoding='cp949')
df.head()

,지역명,연도,월,평당분양가격
0,서울,2013,12,18189.0
1,부산,2013,12,8111.0
2,대구,2013,12,8080.0
3,인천,2013,12,10204.0
4,광주,2013,12,6098.0


- 지역명2 : 지역명필드는 라벨인코딩하여 추가
- 분석대비 원핫인코딩

- 지역명2n : 지역명2의 normalization 스케일 조정하여 추가
- 연도n : 연도의 normalization 스케일 조정하여 추가
- 월n : 월의 normalization 스케일 조정하여 추가
- 평당분양가격n : 평당분양가격의 normalization 스케일 조정하여 추가

- 지역명2s : 지역명2의 standardization 스케일 조정하여 추가
- 연도s : 연도의 standardization 스케일 조정하여 추가
- 월s : 월의 standardization 스케일 조정하여 추가
- 평당분양가격s : 평당분양가격의 standardization 스케일 조정하여 추가

In [78]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

le = LabelEncoder()
df['지역명2'] = le.fit_transform(df['지역명'])

# X, y 분리
X_df = df[['지역명2','연도','월']].values
y_df = df[['평당분양가격']].values

loc_info = df[['지역명','지역명2']].head(17).sort_values(by='지역명2')
loc_column_names = loc_info['지역명'].tolist()
df[loc_column_names] = to_categorical(df['지역명2'])

scaler_x = MinMaxScaler() # X_df를 정규화시킬 객체
df[['지역명2n', '연도n', '월n']] = scaler_x.fit_transform(X_df)
scaler_y = MinMaxScaler() # y_df를 정규화시킬 객체
df['평당분양가격n'] = scaler_y.fit_transform(np.array(y_df).reshape(-1,1))

scaler_x = StandardScaler() 
df[['지역명2s', '연도s', '월s']] = scaler_x.fit_transform(X_df)
scaler_y = StandardScaler()
df['평당분양가격s'] = scaler_y.fit_transform(np.array(y_df).reshape(-1,1))

df.head().T

,0,1,2,3,4
지역명,서울,부산,대구,인천,광주
연도,2013,2013,2013,2013,2013
월,12,12,12,12,12
평당분양가격,18189.0,8111.0,8080.0,10204.0,6098.0
지역명2,8,7,5,11,4
강원,0.0,0.0,0.0,0.0,0.0
경기,0.0,0.0,0.0,0.0,0.0
경남,0.0,0.0,0.0,0.0,0.0
경북,0.0,0.0,0.0,0.0,0.0
광주,0.0,0.0,0.0,0.0,1.0


# 2. 지역명의 라벨 인코딩
- 지역명을 라벨인코딩한 지역명2
- 분석할 경우 원핫인코딩까지 할 것을 추천

In [37]:
from tensorflow.keras.utils import to_categorical # 분류분석시 원핫인코딩
from tensorflow.keras.models import Sequential # 모델 생성
from tensorflow.keras.layers import Dense,Input
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from pandas import Series, DataFrame

In [87]:
# X, y 분리
# X_df = df[['지역명','연도','월']].values
# y_df = df[['평당분양가격']].values
# print(X_df[:3])
# print(y_df[:3])
X_df = df.iloc[:,:-1]
y_df = df.iloc[:,-1]
display(X_df.head())
display(y_df.head())
# 지역명 라벨인코딩, 원핫인코딩
le = LabelEncoder()
X_df['지역명2'] = le.fit_transform(df['지역명'])

# 지역명, 지역명2
loc_info = X_df[['지역명','지역명2']].head(17).sort_values(by='지역명2')
loc_column_names = loc_info['지역명'].tolist()
# print(loc_column_names)

X_df[loc_column_names] = to_categorical(X_df['지역명2'])

# one_hot_encoded =  DataFrame(one_hot_encoded, columns=[f'R{i}' for i in range(len(one_hot_encoded.T))])
# X_df = pd.concat([X_df, one_hot_encoded], axis=1)
X_df.head(5)

,지역명,연도,월,평당분양가격,지역명2,강원,경기,경남,경북,광주,...,제주,충남,충북,지역명2n,연도n,월n,평당분양가격n,지역명2s,연도s,월s
0,서울,2013,12,18189.0,8,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.5000,0.0,1.0,0.328198,0.000000,-1.875367,1.62196
1,부산,2013,12,8111.0,7,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.4375,0.0,1.0,0.065274,-0.204124,-1.875367,1.62196
2,대구,2013,12,8080.0,5,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.3125,0.0,1.0,0.064466,-0.612372,-1.875367,1.62196
3,인천,2013,12,10204.0,11,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.6875,0.0,1.0,0.119878,0.612372,-1.875367,1.62196
4,광주,2013,12,6098.0,4,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.2500,0.0,1.0,0.012757,-0.816497,-1.875367,1.62196


0    1.168591
1   -0.728312
2   -0.734147
3   -0.334363
4   -1.107203
Name: 평당분양가격s, dtype: float64

,지역명,연도,월,평당분양가격,지역명2,강원,경기,경남,경북,광주,...,제주,충남,충북,지역명2n,연도n,월n,평당분양가격n,지역명2s,연도s,월s
0,서울,2013,12,18189.0,8,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.5000,0.0,1.0,0.328198,0.000000,-1.875367,1.62196
1,부산,2013,12,8111.0,7,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.4375,0.0,1.0,0.065274,-0.204124,-1.875367,1.62196
2,대구,2013,12,8080.0,5,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.3125,0.0,1.0,0.064466,-0.612372,-1.875367,1.62196
3,인천,2013,12,10204.0,11,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.6875,0.0,1.0,0.119878,0.612372,-1.875367,1.62196
4,광주,2013,12,6098.0,4,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.2500,0.0,1.0,0.012757,-0.816497,-1.875367,1.62196


# 3. normalization 스케일 조정
- 입력변수(지역명2, 연도, 월)와 타겟변수(평당분양가격) 따로 스케일 조정(MinMaxScaler 이용)
- 지역명2n, 연도n, 월n필드, 평당분양가격n 추가

In [88]:
from sklearn.preprocessing import MinMaxScaler
scaler_x = MinMaxScaler() # x_data를 정규화시킬 객체
X_df['지역명2n'] = scaler_x.fit_transform(np.array(X_df['지역명2']).reshape(-1,1))
X_df['연도n'] = scaler_x.fit_transform(np.array(X_df['연도']).reshape(-1,1))
X_df['월n'] = scaler_x.fit_transform(np.array(X_df['월']).reshape(-1,1))
display(X_df.head(5))
scaler_y = MinMaxScaler()
scaled_y_df = scaler_y.fit_transform(np.array(y_df).reshape(-1,1))
display(scaled_y_df)

,지역명,연도,월,평당분양가격,지역명2,강원,경기,경남,경북,광주,...,제주,충남,충북,지역명2n,연도n,월n,평당분양가격n,지역명2s,연도s,월s
0,서울,2013,12,18189.0,8,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.5000,0.0,1.0,0.328198,0.000000,-1.875367,1.62196
1,부산,2013,12,8111.0,7,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.4375,0.0,1.0,0.065274,-0.204124,-1.875367,1.62196
2,대구,2013,12,8080.0,5,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.3125,0.0,1.0,0.064466,-0.612372,-1.875367,1.62196
3,인천,2013,12,10204.0,11,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.6875,0.0,1.0,0.119878,0.612372,-1.875367,1.62196
4,광주,2013,12,6098.0,4,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.2500,0.0,1.0,0.012757,-0.816497,-1.875367,1.62196


array([[0.32819817],
       [0.06527439],
       [0.06446563],
       ...,
       [0.21439846],
       [0.19941822],
       [0.51684429]])

# 4. standardization 스케일 조정
- 입력변수와 타겟변수 따로 스케일 조정(StandardScaler 이용)
- 지역명2s, 연도s, 월s, 평당분양가격s, 필드 추가

In [65]:
X_df[['지역명2','연도','월']].values

array([[   8, 2013,   12],
       [   7, 2013,   12],
       [   5, 2013,   12],
       ...,
       [   3, 2024,    8],
       [   2, 2024,    8],
       [  14, 2024,    8]], dtype=int64)

In [89]:
from sklearn.preprocessing import StandardScaler
scaler_x = StandardScaler() # x_data를 표준화시킬 객체
X_df['지역명2s'] = scaler_x.fit_transform(np.array(X_df['지역명2']).reshape(-1,1))
X_df['연도s'] = scaler_x.fit_transform(np.array(X_df['연도']).reshape(-1,1))
X_df['월s'] = scaler_x.fit_transform(np.array(X_df['월']).reshape(-1,1))
display(X_df.head(5))
scaler_y = StandardScaler()
scaled_y_df = scaler_y.fit_transform(np.array(y_df).reshape(-1,1))
display(scaled_y_df)

,지역명,연도,월,평당분양가격,지역명2,강원,경기,경남,경북,광주,...,제주,충남,충북,지역명2n,연도n,월n,평당분양가격n,지역명2s,연도s,월s
0,서울,2013,12,18189.0,8,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.5000,0.0,1.0,0.328198,0.000000,-1.875367,1.62196
1,부산,2013,12,8111.0,7,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.4375,0.0,1.0,0.065274,-0.204124,-1.875367,1.62196
2,대구,2013,12,8080.0,5,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.3125,0.0,1.0,0.064466,-0.612372,-1.875367,1.62196
3,인천,2013,12,10204.0,11,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.6875,0.0,1.0,0.119878,0.612372,-1.875367,1.62196
4,광주,2013,12,6098.0,4,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.2500,0.0,1.0,0.012757,-0.816497,-1.875367,1.62196


array([[ 1.16859132],
       [-0.72831216],
       [-0.73414705],
       ...,
       [ 0.34756602],
       [ 0.23948883],
       [ 2.52960734]])